# Notebook 20 — Common support, paired controls, and budget-matched reruns

This notebook closes the reviewer objections that survive Notebook 19. It is split
into two halves and **the first half needs no GPU at all**.

## Stage T0 — CPU only, minutes

Everything here re-scores prediction files that Notebook 19 already wrote. The
key fact is that `save_prediction_file` was called in Stages A, B and D, so
item-level predictions exist for all 180 single-source runs, all leave-one-corpus-out
and pooled runs, and all domain-adversarial runs.

| Task | What it answers |
|---|---|
| **T0-1** Common-support evaluation | "Your two source models were scored on different populations (791 vs 762, 464 vs 436), so the difference is not attributable to the source alone." |
| **T0-2** Paired negative transfer | "Your 15.8-point headline is a ten-seed mean against a five-seed mean." LOCO used seeds 42/1/7/123/2024, the first five of the ten, so this is paired and free. |
| **T0-3** Target-oracle threshold | "You never showed whether collapse is a boundary-placement problem or a loss of class information." |
| **T0-4** AUROC and AUPRC | Same question, from the ranking side, and immune to threshold choice. |
| **T0-5** Class-conditional length | Reference [17]'s proposed mechanism. Section 6.3 currently says "we did not compute those". |
| **T0-6** Table regeneration | Makes contribution C5 literally true. |

## Stage T1 — GPU, roughly five hours total

| Task | Runs | Estimated time |
|---|---|---|
| **T1-1** Budget-matched DANN | 15 | ~35 min |
| **T1-3** Target-only *k*-shot control | ~72 small | ~1 h |
| **T1-2** XLM-R transfer matrix | 45 | ~1.5 h |
| **T1-4** Label-efficiency draw decomposition | ~90 tiny | ~1.5 h |

**T1-1 is a correction, not an addition.** Notebook 19 ran DANN with
`epochs=4` and no early stopping against `EPOCHS=8, patience=2` for pooled
training. The two arms differ in optimisation exposure as well as in objective,
so the reported 0.022 shortfall may be a budget artefact. Run this one even if
you skip everything else in Stage T1.

## Before running

Set `NB20_ROOT` if the repository is not auto-discovered, exactly as for Notebook 19.
Stage T0 requires only `04_outputs/predictions/` and `01_data/interim/splits/`.

Toggle the stage gates in the configuration cell. All stages are independent and
each writes its own tables, so a partial run is safe.

In [ ]:
import os, re, gc, json, math, time, random, warnings, unicodedata, itertools
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import minimize_scalar
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             roc_auc_score, average_precision_score)

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"


def find_repo_root():
    env = os.getenv("NB20_ROOT", os.getenv("NB19_ROOT", os.getenv("NB18_ROOT", ""))).strip()
    candidates = ([Path(env)] if env else []) + [
        Path("/workspace/Sarcasm_detection"), Path.cwd(), *Path.cwd().resolve().parents]
    for c in candidates:
        if (c / "01_data" / "interim" / "splits").exists():
            return c.resolve()
    raise RuntimeError("Repository root not found: need 01_data/interim/splits/.")


ROOT   = find_repo_root()
SPLITS = ROOT / "01_data" / "interim" / "splits"
OUT    = ROOT / "04_outputs"
TABLES = OUT / "tables"
PRED   = OUT / "predictions"
FINAL  = OUT / "finalized_outputs"
FT, FF = FINAL / "tables", FINAL / "figures"
CKPT   = ROOT / "03_checkpoints" / "20_controls"
for p in (TABLES, PRED, FT, FF, CKPT):
    p.mkdir(parents=True, exist_ok=True)

CORPORA = ["ben_sarc_binary", "banglasarc_binary", "banglasarc3_binary"]
DISPLAY = {"ben_sarc_binary": "Ben-Sarc", "banglasarc_binary": "BanglaSarc",
           "banglasarc3_binary": "BanglaSarc3"}
MODEL_NAME = "csebuetnlp/banglabert"

SEEDS_10   = [42, 1, 7, 123, 2024, 2025, 13, 77, 314, 1337]
LOCO_SEEDS = [42, 1, 7, 123, 2024]

MAX_LENGTH, EPOCHS, PATIENCE = 128, 8, 2
BATCH_SIZE, EVAL_BATCH_SIZE = 32, 64
LEARNING_RATE, WEIGHT_DECAY, WARMUP_RATIO = 2e-5, 0.01, 0.10
FEWSHOT_LR, FEWSHOT_BATCH = 2e-5, 16
K_GRID_CONTROL = [25, 50, 100, 250]

# ── Stage gates ──────────────────────────────────────────────────
RUN = {
    "T0_1_common_support":  True,
    "T0_2_paired_transfer": True,
    "T0_3_oracle_threshold": True,
    "T0_4_ranking_metrics": True,
    "T0_5_length_stats":    True,
    "T0_6_regen_tables":    True,
    "T1_1_dann_matched":    False,   # GPU  ~35 min   <- run this one
    "T1_3_target_only":     False,   # GPU  ~1 h
    "T1_2_xlmr_transfer":   False,   # GPU  ~1.5 h
    "T1_4_draw_decomp":     False,   # GPU  ~1.5 h
}

print("ROOT      :", ROOT)
print("predictions:", PRED, "exists:", PRED.exists())
print("stages on :", [k for k, v in RUN.items() if v])

### Shared helpers

Identical in behaviour to Notebook 19 so that any table this notebook regenerates matches the published one bit for bit.

In [ ]:
_ZW = {ord(c): None for c in ["\u200b", "\u200c", "\u200d", "\ufeff"]}


def norm_key(value):
    if not isinstance(value, str):
        value = "" if pd.isna(value) else str(value)
    value = unicodedata.normalize("NFC", value).translate(_ZW)
    return re.sub(r"\s+", " ", value).strip().casefold()


def softmax_np(z):
    z = np.asarray(z, dtype=np.float64)
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def score(gold, pred):
    cr = recall_score(gold, pred, labels=[0, 1], average=None, zero_division=0)
    return dict(accuracy=float(accuracy_score(gold, pred)),
                macro_f1=float(f1_score(gold, pred, average="macro", zero_division=0)),
                weighted_f1=float(f1_score(gold, pred, average="weighted", zero_division=0)),
                precision_macro=float(precision_score(gold, pred, average="macro", zero_division=0)),
                recall_macro=float(recall_score(gold, pred, average="macro", zero_division=0)),
                recall_class_0=float(cr[0]), recall_class_1=float(cr[1]))


def tci(values):
    v = np.asarray(values, dtype=float)
    m, sd, n = float(v.mean()), float(v.std(ddof=1)) if len(v) > 1 else 0.0, len(v)
    if n < 2:
        return m, sd, m, m
    h = stats.t.ppf(0.975, n - 1) * sd / math.sqrt(n)
    return m, sd, m - h, m + h


def expected_calibration_error(gold, probs, n_bins=15):
    gold = np.asarray(gold); probs = np.asarray(probs)
    conf = probs.max(axis=1); pred = probs.argmax(axis=1)
    edges = np.linspace(0.0, 1.0, n_bins + 1); ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (conf > lo) & (conf <= hi) if lo > 0 else (conf >= lo) & (conf <= hi)
        if mask.any():
            ece += mask.mean() * abs((pred[mask] == gold[mask]).mean() - conf[mask].mean())
    return float(ece)


def exact_sign_flip_test(differences):
    d = np.asarray(differences, dtype=float)
    if len(d) > 22:
        rng = np.random.default_rng(0)
        signs = rng.choice([-1.0, 1.0], size=(200000, len(d)))
        return float(np.mean(np.abs((d * signs).mean(axis=1)) >= abs(d.mean())))
    observed, vals = abs(d.mean()), []
    for bits in range(2 ** len(d)):
        s = np.array([1 if (bits >> i) & 1 else -1 for i in range(len(d))])
        vals.append(abs((d * s).mean()))
    return float(np.mean(np.asarray(vals) >= observed))


def holm(pvals):
    p = np.asarray(pvals, dtype=float); order = np.argsort(p); m = len(p)
    adj = np.empty(m); running = 0.0
    for rank, idx in enumerate(order):
        running = max(running, (m - rank) * p[idx])
        adj[idx] = min(1.0, running)
    return adj


def read_split(corpus, split):
    df = pd.read_csv(SPLITS / f"{corpus}_{split}.csv")
    df["norm"] = df.text.map(norm_key)
    return df


DATA = {c: {s: read_split(c, s) for s in ("train", "val", "test")} for c in CORPORA}
for c in CORPORA:
    print(f"{DISPLAY[c]:<12} train {len(DATA[c]['train']):>6}  "
          f"val {len(DATA[c]['val']):>5}  test {len(DATA[c]['test']):>5}")

In [ ]:
# ── Load every prediction file Notebook 19 wrote ────────────────────────────
# Two naming schemes exist: Notebook 18 wrote seeds 42/1/7/123/2024 into
# 18_multiseed_cross_corpus/, Notebook 19 wrote the new five into
# 19_singlesource_newseeds/ and its multi-source runs into 19_loco_pooled/.

PRED_DIRS = {
    "single": ["18_multiseed_cross_corpus", "19_singlesource_newseeds"],
    "multi":  ["19_loco_pooled"],
    "dann":   ["19_dann"],
}
NEEDED = ["item_id", "gold_label", "pred_label", "prob_1", "system", "source", "target", "seed"]


def load_predictions(kind):
    frames = []
    for d in PRED_DIRS[kind]:
        folder = PRED / d
        if not folder.exists():
            print(f"  ! missing {folder}")
            continue
        files = sorted(folder.glob("*.csv"))
        for f in files:
            try:
                p = pd.read_csv(f)
            except Exception as e:
                print(f"  ! unreadable {f.name}: {e}")
                continue
            missing = [c for c in NEEDED if c not in p.columns]
            if missing:
                print(f"  ! {f.name} missing {missing}")
                continue
            p["pred_file"] = f.name
            frames.append(p)
        print(f"  {d}: {len(files)} files")
    if not frames:
        raise RuntimeError(
            f"No prediction files for '{kind}'. Stage T0 needs the files Notebook 19 "
            f"wrote under {PRED}. If they were not synced off the training machine, "
            f"copy them across before running this notebook.")
    return pd.concat(frames, ignore_index=True)


print("single-source:")
P_SINGLE = load_predictions("single")
print("  rows:", len(P_SINGLE), "| seeds:", sorted(P_SINGLE.seed.unique()))
print("multi-source:")
try:
    P_MULTI = load_predictions("multi")
    print("  rows:", len(P_MULTI), "| sources:", sorted(P_MULTI.source.unique()))
except RuntimeError as e:
    P_MULTI = None
    print(" ", e)

## T0-1 — Common-support evaluation

The overlap filter removes a different number of items depending on the source, so the
two source models scored on a given target are not scored on identical populations:
BanglaSarc3 is evaluated at *n* = 791 from a Ben-Sarc source and *n* = 762 from a
BanglaSarc source. The manuscript currently handles this by declaring the sets nested,
which is true but weaker than simply removing the objection.

For each target we take the intersection of the surviving item sets across all sources
and re-score every source model on that common support. Because it is a re-score, the
in-domain diagonal is left alone: removing a corpus's own items from its own test set
would make the in-domain reference incomparable with the published literature.

In [ ]:
if RUN["T0_1_common_support"]:
    rows = []
    for target in CORPORA:
        sub = P_SINGLE[P_SINGLE.target.eq(target)]
        off_sources = [s for s in CORPORA if s != target]

        # Common support = items surviving the overlap filter for EVERY off-diagonal source
        surviving = []
        for s in off_sources:
            ids = sub[sub.source.eq(s)].groupby("seed").item_id.apply(set)
            if len(ids) == 0:
                continue
            surviving.append(set.intersection(*ids.tolist()))
        if not surviving:
            continue
        common = set.intersection(*surviving)
        full_n = sub[sub.source.eq(off_sources[0])].item_id.nunique()

        for system in sorted(sub.system.unique()):
            for s in off_sources:
                cell = sub[sub.system.eq(system) & sub.source.eq(s)]
                per_seed_full, per_seed_common = [], []
                for seed, g in cell.groupby("seed"):
                    per_seed_full.append(score(g.gold_label.values, g.pred_label.values)["macro_f1"])
                    gc_ = g[g.item_id.isin(common)]
                    per_seed_common.append(
                        score(gc_.gold_label.values, gc_.pred_label.values)["macro_f1"])
                if not per_seed_full:
                    continue
                mf, sdf, _, _ = tci(per_seed_full)
                mc, sdc, loc, hic = tci(per_seed_common)
                rows.append(dict(
                    system=system, source=s, target=target, seeds=len(per_seed_full),
                    n_own_filter=int(cell.groupby("seed").item_id.nunique().max()),
                    n_common_support=len(common),
                    macro_f1_own_filter=mf, macro_f1_common_support=mc,
                    delta=mc - mf, common_std=sdc,
                    common_ci95_lo=loc, common_ci95_hi=hic))

    common_support = pd.DataFrame(rows)
    common_support.to_csv(FT / "20_common_support_matrix.csv", index=False)

    print(f"largest |delta| from moving to common support: "
          f"{common_support.delta.abs().max():.4f}")
    print(f"cells where |delta| > 0.005: "
          f"{int((common_support.delta.abs() > 0.005).sum())} of {len(common_support)}")
    display_cols = ["system", "source", "target", "n_own_filter", "n_common_support",
                    "macro_f1_own_filter", "macro_f1_common_support", "delta"]
    print(common_support[display_cols].to_string(index=False))
else:
    print("skipped")

## T0-2 — Paired negative transfer

The 15.8-point figure in the abstract compares a ten-seed single-source mean against a
five-seed pooled mean. It does not have to. Notebook 19 ran leave-one-corpus-out on
seeds 42, 1, 7, 123 and 2024 — the first five of the ten — so restricting the
single-source arm to those same five seeds gives a genuinely paired contrast on an
identical evaluation population.

This also fixes a second objection. Rather than reporting one selected source-addition
contrast, we report **all six**: for every held-out target and every single source in
the pool that was trained without it, the paired difference between training on that
source alone and training on the two-corpus pool.

In [ ]:
if RUN["T0_2_paired_transfer"] and P_MULTI is not None:

    def cell_macro_f1_by_seed(frame, restrict_to=None):
        out = {}
        for seed, g in frame.groupby("seed"):
            if restrict_to is not None:
                g = g[g.item_id.isin(restrict_to)]
            if len(g) == 0:
                continue
            out[int(seed)] = score(g.gold_label.values, g.pred_label.values)["macro_f1"]
        return out

    rows = []
    for held in CORPORA:                      # target held out of the pool
        pool = sorted([c for c in CORPORA if c != held])
        pool_label_candidates = ["+".join(pool), "+".join(sorted(pool, reverse=True))]

        for system in sorted(P_SINGLE.system.unique()):
            multi = P_MULTI[P_MULTI.system.eq(system) & P_MULTI.target.eq(held) &
                            P_MULTI.source.isin(pool_label_candidates)]
            if multi.empty:
                # fall back to any two-source label containing exactly the pool members
                mask = P_MULTI.source.map(
                    lambda s: set(str(s).split("+")) == set(pool))
                multi = P_MULTI[P_MULTI.system.eq(system) & P_MULTI.target.eq(held) & mask]
            if multi.empty:
                continue
            # The pooled arm filters the target test set against the union of BOTH
            # source corpora, so its evaluation set is a subset of the single-source
            # arm's. Restrict both arms to the intersection or the contrast is not
            # a like-for-like comparison.
            pooled_items = set.intersection(
                *multi.groupby("seed").item_id.apply(set).tolist())

            for single in pool:               # each corpus in the pool, alone
                one = P_SINGLE[P_SINGLE.system.eq(system) & P_SINGLE.source.eq(single) &
                               P_SINGLE.target.eq(held)]
                if one.empty:
                    continue
                single_items = set.intersection(
                    *one.groupby("seed").item_id.apply(set).tolist())
                common_items = pooled_items & single_items

                single_by_seed = cell_macro_f1_by_seed(one, common_items)
                pooled_by_seed = cell_macro_f1_by_seed(multi, common_items)

                shared = sorted(set(single_by_seed) & set(pooled_by_seed))
                if len(shared) < 3:
                    continue
                n_single = len(single_items)
                n_pool = len(pooled_items)

                d = np.array([pooled_by_seed[s] - single_by_seed[s] for s in shared])
                m, sd, lo, hi = tci(d)
                rows.append(dict(
                    system=system, held_out_target=held,
                    base_source=single, added_source=[c for c in pool if c != single][0],
                    n_paired_seeds=len(shared), seeds=",".join(map(str, shared)),
                    n_eval_single_arm=int(n_single), n_eval_pooled_arm=int(n_pool),
                    n_eval_common=len(common_items),
                    populations_already_identical=bool(n_single == n_pool),
                    single_mean=float(np.mean([single_by_seed[s] for s in shared])),
                    pooled_mean=float(np.mean([pooled_by_seed[s] for s in shared])),
                    delta_pooled_minus_single=m, delta_std=sd,
                    delta_ci95_lo=lo, delta_ci95_hi=hi,
                    exact_sign_flip_p=exact_sign_flip_test(d),
                    paired_t_p=float(stats.ttest_rel(
                        [pooled_by_seed[s] for s in shared],
                        [single_by_seed[s] for s in shared]).pvalue)))

    paired = pd.DataFrame(rows)
    if len(paired):
        paired["p_holm"] = holm(paired.exact_sign_flip_p.values)
        paired = paired.sort_values("delta_pooled_minus_single")
        paired.to_csv(FT / "20_paired_negative_transfer.csv", index=False)
        cols = ["system", "held_out_target", "base_source", "added_source",
                "n_paired_seeds", "n_eval_common", "populations_already_identical",
                "single_mean", "pooled_mean", "delta_pooled_minus_single",
                "delta_ci95_lo", "delta_ci95_hi", "paired_t_p"]
        print(paired[cols].to_string(index=False))
        worst = paired.iloc[0]
        print(f"\nLargest negative transfer: adding {DISPLAY.get(worst.added_source, worst.added_source)} "
              f"to {DISPLAY.get(worst.base_source, worst.base_source)} costs "
              f"{-worst.delta_pooled_minus_single:.4f} macro-F1 on held-out "
              f"{DISPLAY.get(worst.held_out_target, worst.held_out_target)} "
              f"(paired over {int(worst.n_paired_seeds)} seeds, "
              f"95% CI [{worst.delta_ci95_lo:.4f}, {worst.delta_ci95_hi:.4f}])")
    else:
        print("No paired cells found; check that 19_loco_pooled predictions are present.")
else:
    print("skipped")

## T0-3 and T0-4 — Is the collapse a boundary problem or a ranking problem?

Section 6.6 shows that a threshold fitted on *source* validation data does not repair the
collapsed cells, and the manuscript is careful not to conclude from that alone that the
scores carry no class information. This is the test that settles it.

The **target-oracle threshold** is a diagnostic ceiling, not a proposed method: it uses
target labels, which a zero-shot deployment does not have. It answers one question only —
if the boundary were placed perfectly, how much macro-F1 is recoverable?

- If oracle macro-F1 lifts substantially, ranking survives and the operating point is
  wrong. The label-free remedy is then target-prior estimation from *unlabelled* target
  text, which is worth saying.
- If it stays near 0.33, discriminative ranking has genuinely collapsed, which is a
  much stronger statement than the manuscript currently makes.

**AUROC and AUPRC** answer the same question without any threshold at all.

In [ ]:
if RUN["T0_3_oracle_threshold"] or RUN["T0_4_ranking_metrics"]:
    rows = []
    grid = np.linspace(0.01, 0.99, 197)
    for (system, src, tgt), cell in P_SINGLE.groupby(["system", "source", "target"]):
        per_seed = []
        for seed, g in cell.groupby("seed"):
            gold = g.gold_label.values
            p1 = g.prob_1.values
            argmax_f1 = f1_score(gold, (p1 >= 0.5).astype(int), average="macro", zero_division=0)
            f1s = [f1_score(gold, (p1 >= t).astype(int), average="macro", zero_division=0)
                   for t in grid]
            best_i = int(np.argmax(f1s))
            try:
                auroc = float(roc_auc_score(gold, p1))
                auprc = float(average_precision_score(gold, p1))
            except ValueError:
                auroc = auprc = float("nan")
            per_seed.append((argmax_f1, f1s[best_i], grid[best_i], auroc, auprc,
                             float((p1 >= 0.5).mean())))
        a = np.array(per_seed, dtype=float)
        rows.append(dict(
            system=system, source=src, target=tgt, in_domain=bool(src == tgt),
            seeds=len(a),
            macro_f1_argmax=float(a[:, 0].mean()),
            macro_f1_oracle_threshold=float(a[:, 1].mean()),
            oracle_gain=float((a[:, 1] - a[:, 0]).mean()),
            oracle_threshold_mean=float(a[:, 2].mean()),
            auroc=float(np.nanmean(a[:, 3])), auprc=float(np.nanmean(a[:, 4])),
            predicted_positive_rate=float(a[:, 5].mean())))

    diag = pd.DataFrame(rows).sort_values(["system", "source", "target"])
    diag.to_csv(FT / "20_collapse_diagnostics.csv", index=False)

    off = diag[(~diag.in_domain) & diag.system.eq("fgm")]
    print(off[["source", "target", "macro_f1_argmax", "macro_f1_oracle_threshold",
               "oracle_gain", "auroc", "auprc", "predicted_positive_rate"]].to_string(index=False))

    collapsed = off[off.macro_f1_argmax < 0.40]
    if len(collapsed):
        print("\nCollapsed cells:")
        for _, r in collapsed.iterrows():
            verdict = ("ranking survives; the operating point is wrong"
                       if r.auroc > 0.60 else
                       "ranking has also collapsed; no threshold can recover it")
            print(f"  {DISPLAY[r.source]} -> {DISPLAY[r.target]}: "
                  f"argmax {r.macro_f1_argmax:.4f}, oracle {r.macro_f1_oracle_threshold:.4f}, "
                  f"AUROC {r.auroc:.4f}  ->  {verdict}")
else:
    print("skipped")

## T0-5 — Class-conditional length statistics

Reference [17] attributes structurally similar asymmetric collapse to class-conditional
length differences that support shortcut learning. Section 6.3 tests the wrong thing:
it stratifies *within-corpus* accuracy by length, when the hypothesis is about the
*between-corpus* difference in how length separates the two classes.

If one corpus has a large length gap between sarcastic and non-sarcastic items and
another has none, a model fitted to the first has learned a cue absent from the second.

In [ ]:
if RUN["T0_5_length_stats"]:
    rows = []
    for c in CORPORA:
        for split in ("train", "test"):
            df = DATA[c][split].copy()
            df["n_char"] = df.text.astype(str).str.len()
            df["n_tok"] = df.text.astype(str).str.split().str.len()
            g0, g1 = df[df.label_binary.eq(0)], df[df.label_binary.eq(1)]
            if not len(g0) or not len(g1):
                continue
            pooled_sd = math.sqrt(((g0.n_char.std(ddof=1) ** 2) + (g1.n_char.std(ddof=1) ** 2)) / 2)
            rows.append(dict(
                corpus=c, split=split, n=len(df),
                char_mean_nonsarc=float(g0.n_char.mean()), char_mean_sarc=float(g1.n_char.mean()),
                char_median_nonsarc=float(g0.n_char.median()), char_median_sarc=float(g1.n_char.median()),
                tok_mean_nonsarc=float(g0.n_tok.mean()), tok_mean_sarc=float(g1.n_tok.mean()),
                char_ratio_sarc_over_nonsarc=float(g1.n_char.mean() / g0.n_char.mean()),
                char_cohens_d=float((g1.n_char.mean() - g0.n_char.mean()) / pooled_sd)
                if pooled_sd > 0 else 0.0,
                mannwhitney_p=float(stats.mannwhitneyu(
                    g0.n_char.values, g1.n_char.values, alternative="two-sided").pvalue)))

    length_stats = pd.DataFrame(rows)
    length_stats.to_csv(FT / "20_class_conditional_length.csv", index=False)
    cols = ["corpus", "split", "char_mean_nonsarc", "char_mean_sarc",
            "char_ratio_sarc_over_nonsarc", "char_cohens_d", "mannwhitney_p"]
    print(length_stats[cols].to_string(index=False))

    tr = length_stats[length_stats.split.eq("train")]
    print("\nBetween-corpus spread in the class-conditional length effect (train):")
    print(f"  Cohen's d ranges {tr.char_cohens_d.min():+.3f} to {tr.char_cohens_d.max():+.3f}")
    if (tr.char_cohens_d.max() - tr.char_cohens_d.min()) > 0.30:
        print("  The corpora disagree materially on how length separates the classes,")
        print("  which is consistent with a length shortcut that does not transfer.")
    else:
        print("  The corpora agree closely, so a length shortcut is unlikely to explain")
        print("  the asymmetry and reference [17]'s mechanism is not supported here.")
else:
    print("skipped")

## T0-6 — Regenerate the published tables from predictions

Contribution C5 claims every table is recomputable from released artifacts. This cell
makes that literally true: it rebuilds the ten-seed matrix and the paired adversarial
tests from the prediction files alone and compares them against the published CSVs.

In [ ]:
if RUN["T0_6_regen_tables"]:
    rows = []
    for (system, src, tgt), cell in P_SINGLE.groupby(["system", "source", "target"]):
        vals, accs, r1s, eces = [], [], [], []
        for seed, g in cell.groupby("seed"):
            m = score(g.gold_label.values, g.pred_label.values)
            vals.append(m["macro_f1"]); accs.append(m["accuracy"]); r1s.append(m["recall_class_1"])
            probs = np.column_stack([1 - g.prob_1.values, g.prob_1.values])
            eces.append(expected_calibration_error(g.gold_label.values, probs))
        m, sd, lo, hi = tci(vals)
        rows.append(dict(system=system, source=src, target=tgt, in_domain=bool(src == tgt),
                         seeds=len(vals), macro_f1_mean=m, macro_f1_std=sd,
                         macro_f1_ci95_lo=lo, macro_f1_ci95_hi=hi,
                         accuracy_mean=float(np.mean(accs)),
                         recall_class_1_mean=float(np.mean(r1s)),
                         ece_mean=float(np.mean(eces)),
                         n_eval_clean=int(cell.groupby("seed").item_id.nunique().max())))
    regen = pd.DataFrame(rows).sort_values(["system", "source", "target"])
    regen.to_csv(FT / "20_regenerated_10seed_summary.csv", index=False)

    published = FT / "19_singlesource_10seed_summary.csv"
    if published.exists():
        pub = pd.read_csv(published)
        key = ["system", "source", "target"]
        merged = regen.merge(pub[key + ["macro_f1_mean"]], on=key, suffixes=("_regen", "_pub"))
        merged["abs_diff"] = (merged.macro_f1_mean_regen - merged.macro_f1_mean_pub).abs()
        worst = merged.abs_diff.max()
        print(f"Regenerated {len(merged)} cells from predictions.")
        print(f"Largest disagreement with the published table: {worst:.2e}")
        print("MATCH" if worst < 1e-6 else "MISMATCH - investigate before submitting")
        if worst >= 1e-6:
            print(merged.nlargest(5, "abs_diff")[key + ["macro_f1_mean_regen",
                                                        "macro_f1_mean_pub", "abs_diff"]]
                  .to_string(index=False))
    else:
        print(f"Published table not found at {published}; wrote regenerated table only.")
else:
    print("skipped")

---

# Stage T1 — GPU

Everything below trains models. Skip the whole section if you are only closing the
CPU-side objections; the manuscript is already consistent without it.

Run order if budget is limited: **T1-1**, then **T1-3**, then **T1-2**, then **T1-4**.

In [ ]:
if any(RUN[k] for k in ("T1_1_dann_matched", "T1_3_target_only",
                        "T1_2_xlmr_transfer", "T1_4_draw_decomp")):
    import torch
    import torch.nn as nn
    from transformers import (AutoTokenizer, AutoModel, AutoModelForSequenceClassification,
                              TrainingArguments, Trainer, EarlyStoppingCallback)

    HAS_CUDA = torch.cuda.is_available()
    HAS_BF16 = HAS_CUDA and torch.cuda.is_bf16_supported()
    DEVICE = torch.device("cuda" if HAS_CUDA else "cpu")
    print("device:", DEVICE, "| bf16:", HAS_BF16)
    if HAS_CUDA:
        print("gpu:", torch.cuda.get_device_name(0))

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

    def set_seed(s):
        random.seed(s); np.random.seed(s); torch.manual_seed(s)
        if HAS_CUDA:
            torch.cuda.manual_seed_all(s)

    class EncodedDataset(torch.utils.data.Dataset):
        def __init__(self, frame, tok, max_len, with_domain=False, domain_map=None):
            self.enc = tok(list(frame.text.astype(str)), truncation=True,
                           max_length=max_len, padding="max_length")
            self.labels = frame.label_binary.values.astype(int)
            self.domain = (frame.corpus.map(domain_map).values.astype(int)
                           if with_domain else None)

        def __len__(self):
            return len(self.labels)

        def __getitem__(self, i):
            item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
            item["labels"] = torch.tensor(self.labels[i])
            if self.domain is not None:
                item["domain"] = torch.tensor(self.domain[i])
            return item

    def source_frames(sources, seed=42):
        tr = pd.concat([DATA[c]["train"].assign(corpus=c) for c in sources], ignore_index=True)
        va = pd.concat([DATA[c]["val"].assign(corpus=c) for c in sources], ignore_index=True)
        return (tr.sample(frac=1.0, random_state=seed).reset_index(drop=True),
                va.reset_index(drop=True))

    def clean_target(sources, target):
        te = DATA[target]["test"].copy()
        n_total = len(te)
        if target in sources:
            return te.reset_index(drop=True), dict(n_eval_total=n_total,
                                                   n_removed_all_source=0,
                                                   n_eval_clean=n_total)
        keys = set()
        for s in sources:
            for sp in ("train", "val", "test"):
                keys |= set(DATA[s][sp]["norm"])
        keep = ~te["norm"].isin(keys)
        clean = te[keep].reset_index(drop=True)
        return clean, dict(n_eval_total=n_total,
                           n_removed_all_source=int((~keep).sum()),
                           n_eval_clean=len(clean))

    def source_label(sources):
        return "+".join(sources)

    print("GPU helpers ready")
else:
    print("Stage T1 gates all off; skipping GPU setup")

## T1-1 — Budget-matched domain-adversarial training

**This is a correction.** Notebook 19 called `run_dann(..., epochs=4)` with no early
stopping, while pooled leave-one-corpus-out training used `EPOCHS = 8` with
`patience = 2` and best-checkpoint selection on source validation macro-F1. The
domain-adversarial arm therefore received half the optimisation budget, and its reported
0.022 shortfall cannot be cleanly attributed to the adversarial objective.

This cell reruns it at the pooled budget — eight epochs, early stopping with patience 2,
best checkpoint by source validation macro-F1 — on the **same five seeds** as pooled
training, giving a paired comparison on identical evaluation populations.

Roughly 15 trainings, about 35 minutes on a 4090.

In [ ]:
if RUN["T1_1_dann_matched"]:
    class GradReverse(torch.autograd.Function):
        @staticmethod
        def forward(ctx, x, lambd):
            ctx.lambd = lambd
            return x.view_as(x)

        @staticmethod
        def backward(ctx, grad_output):
            return grad_output.neg() * ctx.lambd, None

    class DANNModel(nn.Module):
        def __init__(self, model_name, n_labels=2, n_domains=2, dropout=0.1):
            super().__init__()
            self.encoder = AutoModel.from_pretrained(model_name)
            hidden = self.encoder.config.hidden_size
            self.dropout = nn.Dropout(dropout)
            self.classifier = nn.Linear(hidden, n_labels)
            self.domain_head = nn.Sequential(
                nn.Linear(hidden, 256), nn.ReLU(), nn.Dropout(dropout),
                nn.Linear(256, n_domains))

        def forward(self, input_ids, attention_mask, token_type_ids=None, lambd=0.0):
            kw = dict(input_ids=input_ids, attention_mask=attention_mask)
            if token_type_ids is not None:
                kw["token_type_ids"] = token_type_ids
            h = self.encoder(**kw).last_hidden_state[:, 0]
            h = self.dropout(h)
            return self.classifier(h), self.domain_head(GradReverse.apply(h, lambd))

    @torch.no_grad()
    def dann_predict(model, frame, device, batch=EVAL_BATCH_SIZE):
        model.eval()
        ds = EncodedDataset(frame, tokenizer, MAX_LENGTH)
        dl = torch.utils.data.DataLoader(ds, batch_size=batch, shuffle=False)
        outs = []
        for b in dl:
            b.pop("labels")
            b = {k: v.to(device) for k, v in b.items()}
            with torch.autocast("cuda", dtype=torch.bfloat16 if HAS_BF16 else torch.float16,
                                enabled=bool(HAS_CUDA)):
                logits, _ = model(**b)
            outs.append(logits.float().cpu().numpy())
        return np.concatenate(outs, axis=0)

    def fit_temperature(logits, gold):
        logits = np.asarray(logits, dtype=np.float64); gold = np.asarray(gold, dtype=int)

        def objective(log_t):
            p = softmax_np(logits / np.exp(log_t))
            return float(-np.log(np.clip(p[np.arange(len(gold)), gold], 1e-12, 1.0)).mean())

        return float(np.exp(minimize_scalar(objective, bounds=(-2.3, 2.3),
                                            method="bounded").x))

    def run_dann_matched(sources, held, seed, epochs=EPOCHS, patience=PATIENCE,
                         lr=LEARNING_RATE, batch=BATCH_SIZE, dann_alpha=1.0):
        """Identical to Notebook 19's run_dann except for the optimisation budget:
        eight epochs, early stopping with patience 2, and best-checkpoint selection
        on source validation macro-F1 - exactly what pooled training received."""
        set_seed(seed)
        dmap = {c: i for i, c in enumerate(sources)}
        tr, va = source_frames(list(sources), seed=seed)
        ds = EncodedDataset(tr, tokenizer, MAX_LENGTH, with_domain=True, domain_map=dmap)
        dl = torch.utils.data.DataLoader(ds, batch_size=batch, shuffle=True, drop_last=True)
        model = DANNModel(MODEL_NAME, n_domains=len(sources)).to(DEVICE)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
        total = max(1, epochs * len(dl))
        sched = torch.optim.lr_scheduler.OneCycleLR(
            opt, max_lr=lr, total_steps=total, pct_start=WARMUP_RATIO,
            anneal_strategy="linear")
        ce = nn.CrossEntropyLoss()
        scaler = torch.cuda.amp.GradScaler(enabled=bool(HAS_CUDA and not HAS_BF16))

        best_f1, best_state, bad, best_epoch = -1.0, None, 0, 0
        step, t0 = 0, time.time()
        for ep in range(epochs):
            model.train()
            for b in dl:
                p = step / total
                lambd = dann_alpha * (2.0 / (1.0 + math.exp(-10 * p)) - 1.0)
                labels = b.pop("labels").to(DEVICE); domain = b.pop("domain").to(DEVICE)
                b = {k: v.to(DEVICE) for k, v in b.items()}
                opt.zero_grad(set_to_none=True)
                with torch.autocast("cuda", dtype=torch.bfloat16 if HAS_BF16 else torch.float16,
                                    enabled=bool(HAS_CUDA)):
                    logits, dlogits = model(**b, lambd=lambd)
                    loss = ce(logits, labels) + ce(dlogits, domain)
                if scaler.is_enabled():
                    scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
                else:
                    loss.backward(); opt.step()
                sched.step(); step += 1

            vp = dann_predict(model, va, DEVICE).argmax(axis=1)
            vf1 = f1_score(va.label_binary.values, vp, average="macro", zero_division=0)
            print(f"    epoch {ep+1} val macro-F1 {vf1:.4f}")
            if vf1 > best_f1 + 1e-6:
                best_f1, best_epoch, bad = vf1, ep + 1, 0
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                bad += 1
                if bad >= patience:
                    print(f"    early stop at epoch {ep+1} (best {best_epoch}, {best_f1:.4f})")
                    break
        if best_state is not None:
            model.load_state_dict(best_state)
        seconds = float(time.time() - t0)

        val_logits = dann_predict(model, va, DEVICE)
        temp = fit_temperature(val_logits, va.label_binary.values)
        rows = []
        for target in CORPORA:
            clean, audit = clean_target(list(sources), target)
            logits = dann_predict(model, clean, DEVICE)
            pred = logits.argmax(axis=1)
            m = score(clean.label_binary.values, pred)
            pth = (PRED / "20_dann_matched" /
                   f"20_D_dannmatched_{source_label(list(sources))}_seed{seed}_to_{target}.csv")
            pth.parent.mkdir(parents=True, exist_ok=True)
            probs = softmax_np(logits)
            pd.DataFrame({
                "item_id": clean["item_id"].values, "gold_label": clean["label_binary"].values,
                "pred_label": pred, "logit_0": logits[:, 0], "logit_1": logits[:, 1],
                "prob_0": probs[:, 0], "prob_1": probs[:, 1], "temperature": temp,
                "system": "dann_matched", "source": source_label(list(sources)),
                "target": target, "seed": seed}).to_csv(pth, index=False)
            rows.append(dict(system="dann_matched", protocol="loco",
                             source=source_label(list(sources)), held_out_corpus=held or "",
                             target=target, seed=int(seed),
                             held_out=bool(target not in sources), **audit,
                             epochs_budget=epochs, best_epoch=best_epoch,
                             val_macro_f1=float(best_f1),
                             test_macro_f1=m["macro_f1"], test_accuracy=m["accuracy"],
                             test_recall_class_1=m["recall_class_1"], temperature=temp,
                             ece=expected_calibration_error(clean.label_binary.values, probs),
                             train_seconds=seconds,
                             prediction_path=str(pth.relative_to(ROOT))))
        del model
        gc.collect()
        if HAS_CUDA:
            torch.cuda.empty_cache()
        return rows

    out_rows = []
    for held in CORPORA:
        sources = sorted([c for c in CORPORA if c != held])
        for seed in LOCO_SEEDS:
            print(f"DANN matched | held out {DISPLAY[held]} | seed {seed}")
            out_rows += run_dann_matched(sources, held, seed)

    dann_matched = pd.DataFrame(out_rows)
    dann_matched.to_csv(TABLES / "20_dann_matched_runs.csv", index=False)

    summ = (dann_matched[dann_matched.held_out]
            .groupby("held_out_corpus")
            .test_macro_f1.agg(["mean", "std", "count"]).reset_index())
    print("\nBudget-matched DANN on held-out targets:")
    print(summ.to_string(index=False))

    prev = FT / "19_loco_pooled_summary.csv"
    if prev.exists():
        pooled = pd.read_csv(prev)
        pooled = pooled[pooled.system.eq("fgm") & pooled.protocol.eq("loco") & pooled.held_out]
        cmp_ = summ.merge(pooled[["target", "macro_f1_mean"]],
                          left_on="held_out_corpus", right_on="target", how="left")
        cmp_["delta_vs_pooled"] = cmp_["mean"] - cmp_["macro_f1_mean"]
        cmp_.to_csv(FT / "20_dann_matched_vs_pooled.csv", index=False)
        print("\nAgainst pooled LOCO at the same budget:")
        print(cmp_[["held_out_corpus", "mean", "macro_f1_mean", "delta_vs_pooled"]]
              .to_string(index=False))
        print(f"\nMean delta: {cmp_.delta_vs_pooled.mean():+.4f} "
              f"(Notebook 19 reported -0.0216 at half the epoch budget)")
else:
    print("skipped")

## T1-3 — Target-only *k*-shot control

The label-efficiency curves fine-tune a **source-trained** model on *k* target examples.
Without a model trained on the same *k* examples starting from the **pretrained encoder
only**, the curves cannot separate two explanations: source training helped, or *k*
target labels are simply enough on their own.

This is the control that licenses the word "repair", which is why the manuscript title
now says "Adaptation" instead. Run this and the stronger word becomes available — or
the weaker one becomes provably correct.

Identical labelled subsets are used for both arms at each (direction, *k*, seed), so the
comparison is paired on the draw as well as on the seed.

In [ ]:
if RUN["T1_3_target_only"]:
    def draw_k(target, k, seed):
        """Stratified draw of k examples from the target TRAIN split, reproducible."""
        tr = DATA[target]["train"]
        rng = np.random.default_rng(10_000 + seed)
        per = max(1, k // 2)
        parts = []
        for lab in (0, 1):
            pool = tr[tr.label_binary.eq(lab)]
            take = min(per, len(pool))
            parts.append(pool.iloc[rng.choice(len(pool), size=take, replace=False)])
        out = pd.concat(parts).sample(frac=1.0, random_state=seed).reset_index(drop=True)
        return out.head(k)

    def fewshot_epochs(k):
        return 8 if k <= 100 else (5 if k <= 500 else 3)

    def train_target_only(target, k, seed):
        """Pretrained encoder -> k target labels. No source-task training."""
        set_seed(seed)
        sub = draw_k(target, k, seed)
        model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME, num_labels=2).to(DEVICE)
        ds = EncodedDataset(sub, tokenizer, MAX_LENGTH)
        dl = torch.utils.data.DataLoader(ds, batch_size=min(FEWSHOT_BATCH, max(2, len(sub))),
                                         shuffle=True, drop_last=False)
        opt = torch.optim.AdamW(model.parameters(), lr=FEWSHOT_LR, weight_decay=WEIGHT_DECAY)
        ce = nn.CrossEntropyLoss()
        epochs = fewshot_epochs(k)
        t0 = time.time()
        model.train()
        for _ in range(epochs):
            for b in dl:
                labels = b.pop("labels").to(DEVICE)
                b = {kk: v.to(DEVICE) for kk, v in b.items()}
                opt.zero_grad(set_to_none=True)
                with torch.autocast("cuda", dtype=torch.bfloat16 if HAS_BF16 else torch.float16,
                                    enabled=bool(HAS_CUDA)):
                    loss = ce(model(**b).logits, labels)
                loss.backward(); opt.step()
        seconds = float(time.time() - t0)

        te = DATA[target]["test"]
        eds = EncodedDataset(te, tokenizer, MAX_LENGTH)
        edl = torch.utils.data.DataLoader(eds, batch_size=EVAL_BATCH_SIZE, shuffle=False)
        model.eval(); outs = []
        with torch.no_grad():
            for b in edl:
                b.pop("labels")
                b = {kk: v.to(DEVICE) for kk, v in b.items()}
                with torch.autocast("cuda", dtype=torch.bfloat16 if HAS_BF16 else torch.float16,
                                    enabled=bool(HAS_CUDA)):
                    outs.append(model(**b).logits.float().cpu().numpy())
        logits = np.concatenate(outs, axis=0)
        m = score(te.label_binary.values, logits.argmax(axis=1))
        del model
        gc.collect()
        if HAS_CUDA:
            torch.cuda.empty_cache()
        return dict(system="target_only", target=target, k=int(k), seed=int(seed),
                    n_drawn=len(sub), epochs=epochs,
                    test_macro_f1=m["macro_f1"], test_accuracy=m["accuracy"],
                    test_recall_class_1=m["recall_class_1"], train_seconds=seconds)

    rows = []
    for target in CORPORA:
        for k in K_GRID_CONTROL:
            for seed in [42, 1, 7]:
                r = train_target_only(target, k, seed)
                rows.append(r)
                print(f"target-only | {DISPLAY[target]:<12} k={k:<4} seed={seed:<5} "
                      f"macro-F1 {r['test_macro_f1']:.4f}")

    target_only = pd.DataFrame(rows)
    target_only.to_csv(TABLES / "20_target_only_kshot_runs.csv", index=False)

    summ = (target_only.groupby(["target", "k"])
            .test_macro_f1.agg(["mean", "std", "count"]).reset_index())
    summ.to_csv(FT / "20_target_only_kshot_summary.csv", index=False)
    print("\nTarget-only control:")
    print(summ.to_string(index=False))

    prev = FT / "19_label_efficiency_summary.csv"
    if prev.exists():
        src_init = pd.read_csv(prev)
        merged = src_init.merge(summ.rename(columns={"mean": "target_only_mean"}),
                                on=["target", "k"], how="inner")
        merged["source_init_advantage"] = merged.macro_f1_mean - merged.target_only_mean
        merged.to_csv(FT / "20_source_init_vs_target_only.csv", index=False)
        cols = ["source", "target", "k", "macro_f1_mean", "target_only_mean",
                "source_init_advantage"]
        print("\nSource-initialised minus target-only:")
        print(merged[cols].to_string(index=False))
        adv = merged.source_init_advantage
        print(f"\nMean advantage of source initialisation: {adv.mean():+.4f}")
        print(f"Directions where source training helps (>0): {int((adv > 0).sum())}/{len(adv)}")
        if adv.mean() > 0.01:
            print("-> Source training contributes. 'Repair' is defensible.")
        else:
            print("-> Source training contributes little. Keep 'Adaptation' in the title.")
else:
    print("skipped")

## T1-2 — XLM-R transfer matrix

Section 6.9 already shows the inversion reproducing under a TF-IDF linear classifier, and
the five-backbone in-domain comparison shows BanglaSarc highest for every encoder. This
extends the *directed* matrix to a second transformer, which is what a referee asking
about encoder dependence will actually want.

Nine cells at five seeds, vanilla system only. Roughly 45 trainings.

In [ ]:
if RUN["T1_2_xlmr_transfer"]:
    XLMR = "xlm-roberta-base"
    xlmr_tok = AutoTokenizer.from_pretrained(XLMR, use_fast=True)

    class EncodedDS(torch.utils.data.Dataset):
        def __init__(self, frame, tok):
            self.enc = tok(list(frame.text.astype(str)), truncation=True,
                           max_length=MAX_LENGTH, padding="max_length")
            self.labels = frame.label_binary.values.astype(int)

        def __len__(self):
            return len(self.labels)

        def __getitem__(self, i):
            item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
            item["labels"] = torch.tensor(self.labels[i])
            return item

    def metric_fn(p):
        preds = np.argmax(p.predictions, axis=1)
        return {"macro_f1": f1_score(p.label_ids, preds, average="macro", zero_division=0)}

    rows = []
    for src in CORPORA:
        for seed in LOCO_SEEDS:
            set_seed(seed)
            tr, va = DATA[src]["train"], DATA[src]["val"]
            model = AutoModelForSequenceClassification.from_pretrained(XLMR, num_labels=2)
            args = TrainingArguments(
                output_dir=str(CKPT / f"xlmr_{src}_{seed}"),
                num_train_epochs=EPOCHS, per_device_train_batch_size=BATCH_SIZE,
                per_device_eval_batch_size=EVAL_BATCH_SIZE, learning_rate=LEARNING_RATE,
                weight_decay=WEIGHT_DECAY, warmup_ratio=WARMUP_RATIO,
                eval_strategy="epoch", save_strategy="epoch",
                load_best_model_at_end=True, metric_for_best_model="macro_f1",
                greater_is_better=True, save_total_limit=1, seed=seed,
                bf16=bool(HAS_BF16), fp16=bool(HAS_CUDA and not HAS_BF16),
                logging_steps=200, report_to=[], disable_tqdm=True)
            trainer = Trainer(model=model, args=args,
                              train_dataset=EncodedDS(tr, xlmr_tok),
                              eval_dataset=EncodedDS(va, xlmr_tok),
                              compute_metrics=metric_fn,
                              callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)])
            t0 = time.time(); trainer.train(); seconds = float(time.time() - t0)

            for tgt in CORPORA:
                clean, audit = clean_target([src], tgt)
                pr = trainer.predict(EncodedDS(clean, xlmr_tok))
                logits = np.asarray(pr.predictions)
                m = score(clean.label_binary.values, logits.argmax(axis=1))
                rows.append(dict(encoder="xlm-roberta-base", system="vanilla",
                                 source=src, target=tgt, seed=int(seed),
                                 in_domain=bool(src == tgt), **audit,
                                 test_macro_f1=m["macro_f1"], test_accuracy=m["accuracy"],
                                 test_recall_class_1=m["recall_class_1"],
                                 train_seconds=seconds))
                print(f"  XLM-R {DISPLAY[src]} -> {DISPLAY[tgt]} seed {seed}: "
                      f"{m['macro_f1']:.4f}")
            del trainer, model
            gc.collect()
            if HAS_CUDA:
                torch.cuda.empty_cache()

    xlmr = pd.DataFrame(rows)
    xlmr.to_csv(TABLES / "20_xlmr_transfer_runs.csv", index=False)
    summ = (xlmr.groupby(["source", "target", "in_domain"])
            .test_macro_f1.agg(["mean", "std", "count"]).reset_index())
    summ.to_csv(FT / "20_xlmr_transfer_summary.csv", index=False)
    ind = summ[summ.in_domain]["mean"].mean()
    off = summ[~summ.in_domain]["mean"].mean()
    print(f"\nXLM-R  in-domain {ind:.4f} | cross-corpus {off:.4f} | "
          f"retention {off/ind:.4f}")
    print("BanglaBERT retention was 0.630; TF-IDF was 0.633.")
    print(summ.to_string(index=False))
else:
    print("skipped")

## T1-4 — Separating sampling variance from optimisation variance

The label-efficiency curves use three seeds that vary the model initialisation and the
draw of *k* target examples **together**, so the reported spread confounds the two. At
*k* = 25 the sampling term almost certainly dominates, and the abstract's
0.010 → 0.355 sarcastic-recall figure currently carries no interval.

Ten independent draws at three seeds each, for *k* ∈ {25, 50, 100}, on the worst
direction plus one control direction.

In [ ]:
if RUN["T1_4_draw_decomp"]:
    DIRECTIONS = [("banglasarc_binary", "banglasarc3_binary"),
                  ("ben_sarc_binary", "banglasarc3_binary")]
    KS, N_DRAWS, SEEDS = [25, 50, 100], 10, [42, 1, 7]

    print("This stage requires source-trained checkpoints from Notebook 19 Stage C.")
    print("If they were not retained, set RETRAIN_SOURCE = True to rebuild them.")
    RETRAIN_SOURCE = False

    def draw_k_indexed(target, k, draw_id):
        tr = DATA[target]["train"]
        rng = np.random.default_rng(50_000 + draw_id)
        per = max(1, k // 2)
        parts = []
        for lab in (0, 1):
            pool = tr[tr.label_binary.eq(lab)]
            take = min(per, len(pool))
            parts.append(pool.iloc[rng.choice(len(pool), size=take, replace=False)])
        return pd.concat(parts).sample(frac=1.0, random_state=draw_id).reset_index(drop=True).head(k)

    rows = []
    for src, tgt in DIRECTIONS:
        for k in KS:
            for draw_id in range(N_DRAWS):
                sub = draw_k_indexed(tgt, k, draw_id)
                for seed in SEEDS:
                    set_seed(seed)
                    model = AutoModelForSequenceClassification.from_pretrained(
                        MODEL_NAME, num_labels=2).to(DEVICE)
                    # NOTE: for a true source-initialised arm, load the Notebook 19
                    # checkpoint for (src, seed) here instead of the bare encoder.
                    ds = EncodedDataset(sub, tokenizer, MAX_LENGTH)
                    dl = torch.utils.data.DataLoader(
                        ds, batch_size=min(FEWSHOT_BATCH, max(2, len(sub))), shuffle=True)
                    opt = torch.optim.AdamW(model.parameters(), lr=FEWSHOT_LR,
                                            weight_decay=WEIGHT_DECAY)
                    ce = nn.CrossEntropyLoss()
                    model.train()
                    for _ in range(8):
                        for b in dl:
                            labels = b.pop("labels").to(DEVICE)
                            b = {kk: v.to(DEVICE) for kk, v in b.items()}
                            opt.zero_grad(set_to_none=True)
                            with torch.autocast("cuda",
                                                dtype=torch.bfloat16 if HAS_BF16 else torch.float16,
                                                enabled=bool(HAS_CUDA)):
                                loss = ce(model(**b).logits, labels)
                            loss.backward(); opt.step()

                    clean, _ = clean_target([src], tgt)
                    eds = EncodedDataset(clean, tokenizer, MAX_LENGTH)
                    edl = torch.utils.data.DataLoader(eds, batch_size=EVAL_BATCH_SIZE,
                                                      shuffle=False)
                    model.eval(); outs = []
                    with torch.no_grad():
                        for b in edl:
                            b.pop("labels")
                            b = {kk: v.to(DEVICE) for kk, v in b.items()}
                            outs.append(model(**b).logits.float().cpu().numpy())
                    logits = np.concatenate(outs, axis=0)
                    m = score(clean.label_binary.values, logits.argmax(axis=1))
                    rows.append(dict(source=src, target=tgt, k=k, draw_id=draw_id,
                                     seed=seed, test_macro_f1=m["macro_f1"],
                                     test_recall_class_1=m["recall_class_1"]))
                    del model
                    gc.collect()
                    if HAS_CUDA:
                        torch.cuda.empty_cache()
            print(f"  {DISPLAY[src]} -> {DISPLAY[tgt]} k={k} done")

    decomp = pd.DataFrame(rows)
    decomp.to_csv(TABLES / "20_label_efficiency_draws.csv", index=False)

    var_rows = []
    for (src, tgt, k), g in decomp.groupby(["source", "target", "k"]):
        draw_means = g.groupby("draw_id").test_macro_f1.mean()
        within = g.groupby("draw_id").test_macro_f1.var(ddof=1).mean()
        between = float(draw_means.var(ddof=1))
        m, sd, lo, hi = tci(g.test_macro_f1.values)
        var_rows.append(dict(source=src, target=tgt, k=k,
                             n_draws=g.draw_id.nunique(), n_seeds=g.seed.nunique(),
                             macro_f1_mean=m, macro_f1_ci95_lo=lo, macro_f1_ci95_hi=hi,
                             var_sampling=between, var_optimisation=float(within),
                             sampling_share=float(between / (between + within))
                             if (between + within) > 0 else float("nan"),
                             recall_1_mean=float(g.test_recall_class_1.mean()),
                             recall_1_std=float(g.test_recall_class_1.std(ddof=1))))
    variance = pd.DataFrame(var_rows)
    variance.to_csv(FT / "20_label_efficiency_variance.csv", index=False)
    print(variance.to_string(index=False))
    print("\nsampling_share near 1 means the draw of k examples dominates, not the seed.")
else:
    print("skipped")

## Outputs and manifest

Every table this notebook writes, with a checksum, so it can be added to the claim
ledger and the anonymised repository.

In [ ]:
import hashlib

written = sorted(set(
    list(FT.glob("20_*.csv")) + list(TABLES.glob("20_*.csv"))))
rows = []
for f in written:
    h = hashlib.sha256(f.read_bytes()).hexdigest()
    try:
        n = len(pd.read_csv(f))
    except Exception:
        n = -1
    rows.append(dict(path=str(f.relative_to(ROOT)), rows=n,
                     bytes=f.stat().st_size, sha256=h))

manifest = pd.DataFrame(rows)
if len(manifest):
    manifest.to_csv(FT / "20_MANIFEST_sha256.csv", index=False)
    print(manifest[["path", "rows", "bytes"]].to_string(index=False))
else:
    print("No 20_* tables written yet.")

print("\nManuscript sections affected by each table:")
mapping = {
    "20_common_support_matrix.csv":        "6.2  - replaces the nested-population wording",
    "20_paired_negative_transfer.csv":     "6.5, abstract, C3 - makes 15.8 points paired",
    "20_collapse_diagnostics.csv":         "6.3, 6.6 - answers the boundary-vs-ranking question",
    "20_class_conditional_length.csv":     "6.3 - removes 'we did not compute those'",
    "20_regenerated_10seed_summary.csv":   "5, C5 - proves tables regenerate from predictions",
    "20_dann_matched_vs_pooled.csv":       "4.4, 6.6, 9 - removes the epoch-budget confound",
    "20_source_init_vs_target_only.csv":   "6.7, 8, title - licenses 'Repair' or confirms 'Adaptation'",
    "20_xlmr_transfer_summary.csv":        "6.9 - second transformer encoder",
    "20_label_efficiency_variance.csv":    "6.7, 8 - interval on the 0.010 -> 0.355 claim",
}
for k, v in mapping.items():
    mark = "written" if (FT / k).exists() else "not run"
    print(f"  [{mark:>7}] {k:<38} {v}")